# 9단계 | 원본 PDF 정답 구축과 정답 기반 통합 평가

- 개발용 대표 P/N 3개 × 필수 규격 6개 대조함
- 제조사 PDF를 먼저 시각 대조하고 정답 CSV를 확정한 뒤 저장 응답과 비교함
- PDF 실제 쪽과 인쇄 쪽을 별도 기록함

In [1]:
from collections import Counter
from decimal import Decimal
from pathlib import Path
import hashlib
import json
import os
import re

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import pymupdf
import torch
from PIL import Image
from IPython.display import display

from table_detection import load_table_detector, load_structure_recognizer
from datasheet_analysis import (
    SPEC_FIELDS, detect_page_regions, extract_document_source, normalize_spec_value,
    normalize_whitespace, recognize_region_structure, select_extraction_evidence,
)

ROOT = Path.cwd()
RESULT_ROOT = ROOT
targets = (
    ('vishay_t55', 'T55A226M2R5C0200'),
    ('kyocera_avx_tcj', 'TCJU337M016R0050E'),
    ('panasonic_tpb', '4TPB68M'),
)

def read_csv(path):
    """빈 값을 보존한 채 CSV를 읽음."""
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')

manifest = read_csv(ROOT/'data/datasheet_manifest.csv').set_index('document_id',verify_integrity=True)
parts = read_csv(ROOT/'data/part_numbers.csv').set_index(['document_id','part_number'],verify_integrity=True)
splits = read_csv(ROOT/'data/dataset_split.csv').set_index('document_id',verify_integrity=True)
truth_path = ROOT/'data/ground_truth.csv'
truth_sha256 = hashlib.sha256(truth_path.read_bytes()).hexdigest()
truth = read_csv(truth_path)
assert len(truth)==18 and not truth.duplicated(['document_id','part_number','spec_name']).any()
assert set(zip(truth.document_id,truth.part_number,truth.spec_name)) == {
    (*target,field) for target in targets for field in SPEC_FIELDS}
assert truth.status.isin(['found','not_found','ambiguous']).all()
assert truth.review_status.isin(['confirmed','needs_review','unresolved']).all()
assert truth.loc[truth.status.eq('found'),['value','unit','scope','source_type','pdf_page','evidence_quote']].ne('').all().all()
assert truth.loc[truth.status.ne('found'),['value','unit']].eq('').all().all()
assert truth.loc[truth.spec_name.eq('esr'), 'limit_kind'].eq('max').all()
assert truth.loc[truth.spec_name.eq('operating_temperature'),['value_min','value_max']].ne('').all().all()
assert truth.loc[truth.status.eq('ambiguous'),'review_status'].eq('needs_review').all()

paths = {doc:ROOT/'data/datasheets'/manifest.loc[doc,'filename'] for doc,_ in targets}
visible_pages = {}
for (doc,pn), group in truth.groupby(['document_id','part_number'],sort=False):
    assert manifest.loc[doc,'split']==parts.loc[(doc,pn),'split']==splits.loc[doc,'split']=='dev'
    assert group.dataset_split.eq('dev').all()
    assert group.manufacturer.eq(manifest.loc[doc,'manufacturer']).all()
    assert group.series.eq(manifest.loc[doc,'series']).all()
    assert group.pdf_sha256.eq(hashlib.sha256(paths[doc].read_bytes()).hexdigest()).all()
    with pymupdf.open(paths[doc]) as pdf:
        assert len(pdf)==int(manifest.loc[doc,'page_count'])
        for row in group.itertuples():
            pages = [int(n) for n in row.pdf_page.split(';')]
            refs = json.loads(row.evidence_items)
            assert pages==sorted({ref['pdf_page'] for ref in refs})
            assert row.evidence_quote==' || '.join(ref['quote'] for ref in refs)
            for page_no,printed in zip(pages,row.printed_page.split(';'),strict=True):
                assert 1<=page_no<=len(pdf)
                page = pdf[page_no-1]
                foot = {w[4] for w in page.get_text('words') if w[1]>page.rect.height-90}
                assert page.get_label().endswith(printed) or printed in foot
                if (doc,page_no) not in visible_pages:
                    hidden = [pymupdf.Rect(t['bbox']) for t in page.get_texttrace()
                              if t['type']==3 or t.get('opacity')==0]
                    visible_pages[doc,page_no] = ' '.join(w[4] for w in page.get_text('words')
                        if not any(box.contains(pymupdf.Point((w[0]+w[2])/2,(w[1]+w[3])/2)) for box in hidden))
            for ref in refs:
                assert ref['quote'] in visible_pages[doc,ref['pdf_page']]
display(truth[['manufacturer','part_number','spec_name','status','value','unit','condition','pdf_page','printed_page','review_status']])
print('원본 PDF 대조와 정답 CSV 재읽기 확인함.',dict(Counter(truth.review_status)))
print('비교 전에 확보한 정답 SHA-256:',truth_sha256)

C:\Users\human-20\AppData\Local\Temp\ipykernel_10856\3579610613.py:42: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  manifest = read_csv(ROOT/'data/datasheet_manifest.csv').set_index('document_id',verify_integrity=True)
C:\Users\human-20\AppData\Local\Temp\ipykernel_10856\3579610613.py:43: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  parts = read_csv(ROOT/'data/part_numbers.csv').set_index(['document_id','part_number'],verify_integrity=True)
C:\Users\human-20\AppData\Local\Temp\ipykernel_10856\3579610613.py:44: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  splits = read_csv(ROOT/'data/dataset_spl

,manufacturer,part_number,spec_name,status,value,unit,condition,pdf_page,printed_page,review_status
0,Vishay,T55A226M2R5C0200,rated_voltage,found,2.5,VDC,,5,5,confirmed
1,Vishay,T55A226M2R5C0200,surge_voltage,ambiguous,,,,5;15,5;15,needs_review
2,Vishay,T55A226M2R5C0200,capacitance,found,22,μF,,5,5,confirmed
3,Vishay,T55A226M2R5C0200,tolerance,found,± 20,%,,1;5,1;5,confirmed
4,Vishay,T55A226M2R5C0200,esr,found,200,mΩ,+25 °C; 100 kHz,5,5,confirmed
5,Vishay,T55A226M2R5C0200,operating_temperature,found,-55 °C to +105 °C,°C,,1;5,1;5,confirmed
6,KYOCERA AVX,TCJU337M016R0050E,rated_voltage,found,16,V,,6,181,confirmed
7,KYOCERA AVX,TCJU337M016R0050E,surge_voltage,not_found,,,,6;10,181;185,confirmed
8,KYOCERA AVX,TCJU337M016R0050E,capacitance,found,330,μF,,6,181,confirmed
9,KYOCERA AVX,TCJU337M016R0050E,tolerance,found,±20,%,,1;6,176;181,confirmed


원본 PDF 대조와 정답 CSV 재읽기 확인함. {'confirmed': 17, 'needs_review': 1}
비교 전에 확보한 정답 SHA-256: 91b07f80b209809751f35e296a5ca39edd0742eee987fe23cdfe68a3c62a2970


## 저장 Gemini 응답과 평가 범위

- 실제 HTTP 200 응답 2건의 원문으로 평가함
- Panasonic은 정답만 구축하고 모델 응답이 없어 제외함

In [2]:
stored = json.loads((ROOT/'results/07_llm_extraction.json').read_bytes())
bundle = json.loads((ROOT/'results/06_llm_input_samples.json').read_bytes())
prior = stored['remaining_pn_run']
vishay, = [r for r in stored['responses'] if r['part_number']==targets[0][1] and r['http_status']==200]
kyocera, = [r for r in prior['execution_records'] if r['part_number']==targets[1][1] and r['http_status']==200]
responses = {(r['document_id'],r['part_number']):json.loads(r['raw_response_text']) for r in (vishay,kyocera)}
assert set(responses)==set(targets[:2])
assert hashlib.sha256(vishay['raw_response_text'].encode()).hexdigest()==stored['independent_validation']['raw_response_sha256']
stage7 = read_csv(ROOT/'results/07_llm_extraction_summary.csv').set_index(
    ['document_id','part_number','specification'],verify_integrity=True)
assert len(stage7)==12
contexts = {}
for key,response in responses.items():
    doc,pn = key
    assert response['part_number']==pn and set(response['specifications'])==set(SPEC_FIELDS)
    supplemental = (extract_document_source(paths[doc],doc,[10,11,12],[])[0]
                    if doc=='kyocera_avx_tcj' else None)
    contexts[key] = select_extraction_evidence(bundle,doc,pn,supplemental)
    for field,raw in response['specifications'].items():
        record = stage7.loc[(*key,field)]
        assert record.extraction_status==raw['status'] and record.value==(raw['value'] or '')
        assert record.unit==(raw['unit'] or '') and record.condition==(raw['condition'] or '')
panasonic_calls = [r for r in [*prior['execution_records'],stored['panasonic_retry_run']['execution_record']]
                   if r['part_number']==targets[2][1]]
assert len(panasonic_calls)==2 and all(r['http_status']==503 and not r.get('raw_response_text') for r in panasonic_calls)
print('저장된 실제 응답 2건의 규격 12개와 Panasonic 미응답 이력 확인함.')

C:\Users\human-20\AppData\Local\Temp\ipykernel_10856\3297830027.py:9: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  stage7 = read_csv(ROOT/'results/07_llm_extraction_summary.csv').set_index(


저장된 실제 응답 2건의 규격 12개와 Panasonic 미응답 이력 확인함.


## 값·단위·조건·적용 범위·인용 비교

- 단위 변환 후 10진수 완전 일치 기준 적용함
- 동작 온도는 상한·하한을 각각 비교함
- 정량값 없는 규격은 상태 일치로 값 지표를 판정하되, 완전 일치에는 근거 필요함
- ESR 조건은 온도와 주파수를 각각 정규화해 비교함
- 인용 문구·출처 유형과 표의 행·열 또는 시리즈 적용을 함께 확인함
- P/N 적용 근거가 없어 판정 불가인 경우 P/N 지표 분모에서 제외함

In [3]:
def normalized(field, value, unit, low=None, high=None):
    """비교 가능한 원문 수치·단위를 기존 변환 함수로 정규화함."""
    if not value or not unit:
        return None
    try:
        return normalize_spec_value(field,value,unit,low or None,high or None)
    except ValueError:
        return None

def numeric_key(value):
    """변환 후 10진수 값과 단위로 비교함. 허용 오차는 0임."""
    if value is None:
        return None
    if 'min' in value:
        return value['unit'],Decimal(value['min']),Decimal(value['max'])
    text = value['value']
    return value['unit'],text.startswith('±'),Decimal(text.removeprefix('±'))

def condition_terms(condition):
    """온도와 주파수를 각각 °C와 Hz로 정규화함."""
    terms = {}
    cleaned = re.sub(r'\s+', '', condition or '').replace('º','°').replace('−','-').replace('–','-')
    for number,unit in re.findall(r'([+-]?\d+(?:\.\d+)?)(°C|MHz|kHz|Hz)',cleaned):
        if unit=='°C':
            terms.setdefault('temperature',set()).add(Decimal(number))
        else:
            terms.setdefault('frequency',set()).add(Decimal(number)*{'Hz':1,'kHz':1000,'MHz':1000000}[unit])
    return terms

# 시각 대조한 원문 근거의 행·열 연결 기준을 지정함
table_anchors = {
    'vishay_t55': {'capacitance':'CAPACITANCE (μF)', 'esr':'MAX. ESR AT + 25 °C 100 kHz (mΩ)'},
    'kyocera_avx_tcj': {'rated_voltage':'Rated Voltage (V)', 'capacitance':'Capacitance (μF)',
                       'esr':'ESR Max. @ 100kHz (mΩ)', 'operating_temperature':'Maximum Operating Temperature (ºC)'},
}
tolerance_quotes = {
    'vishay_t55': ('series_electrical_context','Capacitance Tolerance: ± 20 %'),
    'kyocera_avx_tcj': ('ordering_guide','M = ±20%'),
}

def citation_checks(key, raw):
    """실제 전송 근거와 PDF의 가시 원문을 모두 만족하는 인용만 남김."""
    cited = [(raw['page'],raw['evidence_id'],raw['evidence'])]
    cited.extend((ref['page'],ref['evidence_id'],ref['quote']) for ref in raw['supporting_evidence'])
    entries = {ref['evidence_id']:ref for ref in contexts[key]['evidence']}
    valid = []
    for page,eid,quote in cited:
        entry = entries.get(eid)
        if not (quote and entry and entry['pdf_page']==page):
            continue
        folded = normalize_whitespace(quote)
        pattern = r'(?<!\w)'+re.escape(folded)+r'(?!\w)'
        if re.search(pattern,entry['text']) and re.search(pattern,visible_pages.get((key[0],page),'')):
            valid.append((entry,folded))
    return valid,len(valid)==len(cited)

def supports_scope(t, raw, citations):
    """원문 행·헤더 또는 주문 코드로 P/N 적용 범위를 확인함."""
    if raw['status']!='found':
        return None
    if raw['scope']!=t.scope:
        return False
    if t.spec_name=='tolerance':
        label,quote = tolerance_quotes[t.document_id]
        return True if any(e['label']==label and quote in q for e,q in citations) else None
    anchor = table_anchors.get(t.document_id,{}).get(t.spec_name)
    if anchor:
        value = raw['value_max'] or raw['value'] if t.spec_name=='operating_temperature' else raw['value']
        row_ok = any(e['label']=='exact_target_pn_row' and t.part_number in e['text'] and value
                     and re.search(r'(?<![\w.])'+re.escape(value)+r'(?![\w.])',q) for e,q in citations)
        header_ok = any(e['label']=='table_header' and anchor in q for e,q in citations)
        return True if row_ok and header_ok else None
    if (t.document_id,t.spec_name)==('vishay_t55','rated_voltage'):
        return True if any(e['label']=='voltage_group' and '2.5 VDC' in q for e,q in citations) else None
    return None

detail_rows, comparison_errors = [], []
for t in truth.itertuples():
    key = t.document_id,t.part_number
    raw = responses[key]['specifications'][t.spec_name] if key in responses else {}
    expected = normalized(t.spec_name,t.value,t.unit,t.value_min,t.value_max)
    predicted = normalized(t.spec_name,raw.get('value'),raw.get('unit'),raw.get('value_min'),raw.get('value_max'))
    score = dict.fromkeys(('status_match','value_match','condition_match','pn_match','page_match',
                           'quote_match','source_type_match','evidence_match','full_match'))
    diagnostics = []
    if not raw:
        evaluation_status = 'not_evaluated_no_response'
    elif t.review_status!='confirmed' or t.status=='ambiguous':
        evaluation_status = 'ground_truth_uncertain'
        diagnostics.append(('ground_truth_unit_uncertain','원문 서지 수치의 단위를 확정하지 못해 채점에서 제외함.'))
        if raw['status']=='found' and not raw['unit']:
            diagnostics.append(('prediction_unit_missing','예측 found 응답의 필수 단위가 비어 있음. 정답 미확정으로 오답 집계에서 제외함.'))
    else:
        evaluation_status = 'evaluated'
        score['status_match'] = raw['status']==t.status
        if t.status=='not_found':
            score['value_match'] = score['status_match']
        else:
            assert expected is not None
            score['value_match'] = raw['status']=='found' and numeric_key(predicted)==numeric_key(expected)
        if t.condition:
            expected_terms = condition_terms(t.condition)
            assert expected_terms
            score['condition_match'] = condition_terms(raw['condition'])==expected_terms
        citations,quote_ok = citation_checks(key,raw)
        pages = {int(n) for n in t.pdf_page.split(';')}
        score['page_match'] = raw['page'] in pages and all(ref['page'] in pages for ref in raw['supporting_evidence'])
        score['quote_match'] = quote_ok
        score['source_type_match'] = bool(citations and raw['source_type']==citations[0][0]['source_type']==t.source_type)
        score['pn_match'] = supports_scope(t,raw,citations)
        # 상한만 인용한 경우 해당 P/N의 전체 온도 범위 근거로 인정하지 않음
        score['evidence_match'] = all(score[m] is True for m in ('page_match','quote_match','source_type_match','pn_match'))
        if t.spec_name=='operating_temperature' and not score['value_match']:
            score['evidence_match'] = False
        score['full_match'] = all(score[m] is True for m in ('status_match','value_match','pn_match','evidence_match')) and score['condition_match'] is not False

        if score['value_match'] is False:
            if t.spec_name=='operating_temperature':
                if not raw['value_min']:
                    diagnostics.append(('temperature_lower_missing','동작 온도 하한이 누락됨.'))
                if raw['value_max'] and raw['value_max']!=t.value_max:
                    diagnostics.append(('temperature_upper_wrong','대상 P/N의 상한과 시리즈 상한이 혼재함.'))
            elif t.status=='found' and not raw['unit']:
                diagnostics.append(('prediction_unit_missing','확정된 값의 필수 단위가 누락됨.'))
            else:
                diagnostics.append(('status_or_value_mismatch','정답 상태 또는 정규화 물리량이 일치하지 않음.'))
        if score['condition_match'] is False:
            diagnostics.append(('measurement_condition_missing','ESR 기준 온도 또는 주파수가 정답 조건과 일치하지 않음.'))
        if score['pn_match'] is False:
            diagnostics.append(('scope_mismatch','대상 P/N에 적용되는 규격을 시리즈 공통값으로 분류함.'))
        elif score['pn_match'] is None:
            diagnostics.append(('pn_link_unconfirmed','인용이 없어 해당 P/N의 카테고리와 시험식 확인 경로를 검증할 수 없음.'))
        if not raw['evidence']:
            diagnostics.append(('evidence_missing','정량값이 없다는 상태는 일치하나, PDF 페이지와 시험식 인용은 누락됨.'))
        elif score['source_type_match'] is False:
            diagnostics.append(('source_type_mismatch','정답의 P/N 규격을 지지하는 표 근거와 응답의 근거 유형이 일치하지 않음.'))
        elif score['evidence_match'] is False:
            diagnostics.append(('range_evidence_incomplete','대상 P/N 행의 상한은 인용했으나 전체 범위를 제시하는 카테고리 근거가 누락됨.'))

    stage7_status = stage7.loc[(*key,t.spec_name),'verification_status'] if raw else None
    detail_rows.append(dict(document_id=t.document_id,manufacturer=t.manufacturer,part_number=t.part_number,
        spec_name=t.spec_name,dataset_split=t.dataset_split,ground_truth_status=t.status,review_status=t.review_status,
        prediction_status=raw.get('status'),ground_truth_value=t.value,prediction_value=raw.get('value'),
        ground_truth_unit=t.unit,prediction_unit=raw.get('unit'),
        ground_truth_normalized=json.dumps(expected,ensure_ascii=False) if expected else None,
        prediction_normalized=json.dumps(predicted,ensure_ascii=False) if predicted else None,
        ground_truth_value_min=t.value_min,prediction_value_min=raw.get('value_min'),
        ground_truth_value_max=t.value_max,prediction_value_max=raw.get('value_max'),
        ground_truth_condition=t.condition,prediction_condition=raw.get('condition'),
        ground_truth_scope=t.scope,prediction_scope=raw.get('scope'),
        ground_truth_source_type=t.source_type,prediction_source_type=raw.get('source_type'),
        ground_truth_page=t.pdf_page,prediction_page=str(raw['page']) if raw.get('page') is not None else None,
        ground_truth_printed_page=t.printed_page,ground_truth_evidence=t.evidence_quote,
        prediction_evidence=raw.get('evidence'),
        prediction_supporting_evidence=json.dumps(raw['supporting_evidence'],ensure_ascii=False) if raw else None,
        ground_truth_test_context=t.test_context,stage7_verification_status=stage7_status,
        **score,evaluation_status=evaluation_status))
    for issue,message in diagnostics:
        comparison_errors.append(dict(manufacturer=t.manufacturer,document_id=t.document_id,part_number=t.part_number,
            spec_name=t.spec_name,category='stage9_comparison' if evaluation_status=='evaluated' else 'stage9_excluded',
            error_type=issue,ground_truth=t.value or t.status,prediction=raw['value'] or raw['status'],
            evidence=t.evidence_quote,description=message,confirmation_status=evaluation_status))
detail = pd.DataFrame(detail_rows)
assert len(detail)==18 and not detail.duplicated(['document_id','part_number','spec_name']).any()
assert detail.evaluation_status.value_counts().to_dict()=={'evaluated':11,'not_evaluated_no_response':6,'ground_truth_uncertain':1}
display(detail[['manufacturer','spec_name','ground_truth_status','prediction_status','value_match','condition_match',
                'pn_match','evidence_match','full_match','evaluation_status']])

,manufacturer,spec_name,ground_truth_status,prediction_status,value_match,condition_match,pn_match,evidence_match,full_match,evaluation_status
0,Vishay,rated_voltage,found,found,True,None,False,False,False,evaluated
1,Vishay,surge_voltage,ambiguous,found,None,None,None,None,None,ground_truth_uncertain
2,Vishay,capacitance,found,found,True,None,True,True,True,evaluated
3,Vishay,tolerance,found,found,True,None,True,True,True,evaluated
4,Vishay,esr,found,found,True,True,True,True,True,evaluated
5,Vishay,operating_temperature,found,found,False,None,False,False,False,evaluated
6,KYOCERA AVX,rated_voltage,found,found,True,None,True,True,True,evaluated
7,KYOCERA AVX,surge_voltage,not_found,not_found,True,None,None,False,False,evaluated
8,KYOCERA AVX,capacitance,found,found,True,None,True,True,True,evaluated
9,KYOCERA AVX,tolerance,found,found,True,None,True,True,True,evaluated


## 분자·분모와 오류 분석

- 정답 확정 11개 규격만 비교함
- 7단계 진단 및 과거 API 오류와 9단계 비교 결과를 구분함
- 백분율은 소수 둘째 자리까지 반올림함

In [4]:
metrics = {
    'value_and_unit_accuracy':'value_match','condition_accuracy':'condition_match',
    'pn_match_accuracy':'pn_match','evidence_accuracy':'evidence_match','full_spec_accuracy':'full_match',
}
summary_rows = []
def summarize(group,item,subset):
    """판정 가능한 셀만 분모로 계산함."""
    counts = subset.evaluation_status.value_counts()
    for metric,col in metrics.items():
        values = subset[col].dropna()
        correct = int(values.sum()) if len(values) else None
        summary_rows.append(dict(group=group,item=item,metric=metric,correct=correct,denominator=len(values),
            accuracy_percent=round(correct*100/len(values),2) if len(values) else None,
            planned=len(subset),response_rows=int(subset.prediction_status.notna().sum()),
            evaluated=int(counts.get('evaluated',0)),ground_truth_uncertain=int(counts.get('ground_truth_uncertain',0)),
            no_response=int(counts.get('not_evaluated_no_response',0))))

summarize('overall','development_sample',detail)
for field in SPEC_FIELDS:
    summarize('specification',field,detail.loc[detail.spec_name.eq(field)])
for maker,rows in detail.groupby('manufacturer',sort=False):
    summarize('manufacturer',maker,rows)
summary = pd.DataFrame(summary_rows)
summary['correct'] = summary['correct'].astype('Int64')
overall = summary.loc[summary.group.eq('overall')].set_index('metric')
assert {m:(int(overall.loc[m,'correct']),int(overall.loc[m,'denominator'])) for m in metrics} == {
    'value_and_unit_accuracy':(9,11),'condition_accuracy':(1,2),'pn_match_accuracy':(8,10),
    'evidence_accuracy':(7,11),'full_spec_accuracy':(6,11),
}
for metric in metrics:
    record = overall.loc[metric]
    per_field = summary.loc[summary.group.eq('specification') & summary.metric.eq(metric)]
    assert per_field.denominator.sum()==record.denominator and per_field.correct.sum()==record.correct
display(overall[['correct','denominator','accuracy_percent']])

old_errors = read_csv(ROOT/'results/07_llm_extraction_errors.csv')
errors = pd.DataFrame(comparison_errors)
historic = old_errors.rename(columns={'specification':'spec_name','pdf_check':'evidence','reason':'description'})
historic = historic.assign(
    category=historic.category.map(lambda value:'stage7_diagnostic' if value=='specification' else 'api'),
    ground_truth='',prediction=historic.raw_field,
    confirmation_status=historic.category.map(lambda value:'prior_pdf_or_schema_review' if value=='specification' else 'recorded_http_503'),
)
errors = pd.concat([errors,historic[errors.columns]],ignore_index=True)
issues = errors.loc[errors.category.eq('stage9_comparison')]
api = errors.loc[errors.category.eq('api')]
assert not issues.duplicated(['document_id','part_number','spec_name','error_type']).any()
assert len(api)==4 and api.error_type.eq('http_503').all()
assert len(errors.loc[errors.category.eq('stage7_diagnostic')])==12
print('정답 비교 사유:',len(issues),'건 | 완전 일치 실패:',int(detail.full_match.eq(False).sum()),'규격 | 이전 HTTP 503:',len(api),'건')

,correct,denominator,accuracy_percent
metric,,,
value_and_unit_accuracy,9,11,81.82
condition_accuracy,1,2,50.00
pn_match_accuracy,8,10,80.00
evidence_accuracy,7,11,63.64
full_spec_accuracy,6,11,54.55


정답 비교 사유: 10 건 | 완전 일치 실패: 5 규격 | 이전 HTTP 503: 4 건


## 기존 DL 탐지·구조 인식 회귀 검사

In [5]:
paths={'vishay_t55': ROOT/'data/datasheets'/manifest.loc['vishay_t55','filename']}
bundle=json.loads((ROOT/'results/06_llm_input_samples.json').read_bytes())
torch.set_num_threads(min(4,os.cpu_count() or 1))
cache=ROOT/'.venv/model_cache'
pr,mo,de,_=load_table_detector(cache_dir=cache,local_files_only=True)
detector=(pr,mo,de)
pr,mo,de,_=load_structure_recognizer(cache_dir=cache,local_files_only=True)
recognizer=(pr,mo,de)
image,meta,regions,_=detect_page_regions(paths['vishay_t55'],5,detector)
prior=bundle['page_sources']['vishay_t55:p5']
assert len(regions)==len(prior['table_regions'])==1
for region,saved in zip(regions,prior['table_regions']):
    assert np.allclose(region['expanded_bbox_pdf'],saved['expanded_bbox_pdf'],atol=.05,rtol=0)
    assert abs(region['confidence']-saved['confidence'])<1e-6
    current=recognize_region_structure(image,meta,region,recognizer)['objects']
    previous=prior['row_column_candidates'][0]['objects']
    assert len(current)==len(previous)
    assert all(a['label']==b['label'] and abs(a['confidence']-b['confidence'])<1e-6 and
        np.allclose(a['bbox_pdf'],b['bbox_pdf'],atol=.05,rtol=0) for a,b in zip(current,previous))
print('기존 탐지·구조 좌표와 점수 일치함')

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

[transformers] TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-detection
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

[transformers] TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-structure-recognition
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


기존 탐지·구조 좌표와 점수 일치함


## 정확도와 미평가 상태 시각화

In [6]:
font = Path('C:/Windows/Fonts/malgun.ttf')
font_manager.fontManager.addfont(font)
plt.rcParams.update({'font.family':font_manager.FontProperties(fname=font).get_name(),'axes.unicode_minus':False})
labels = ['정격전압','서지전압','정전용량','허용오차','ESR','동작 온도']
makers = [manifest.loc[doc,'manufacturer'] for doc,_ in targets]

fig, ax = plt.subplots(figsize=(10,4.9),layout='constrained')
selected = summary.loc[summary.group.eq('specification') & summary.metric.eq('full_spec_accuracy')]
positions = np.arange(len(labels))
bars = ax.barh(positions, selected.accuracy_percent, color='#346a82',height=.56)
ax.bar_label(bars,labels=[f'{int(r.correct)}/{r.denominator}  ({r.accuracy_percent:g}%)'
                         for r in selected.itertuples()],padding=5)
total = overall.loc['full_spec_accuracy']
ax.set(title=f'규격별 완전 일치 정확도 · 전체 {int(total.correct)}/{int(total.denominator)} ({total.accuracy_percent:.2f}%)',
       yticks=positions,yticklabels=labels,xlim=(0,122),xticks=range(0,101,20),
       xlabel='완전 일치율 (%) · 개발용 표본, 판정 가능 분모만 반영')
ax.invert_yaxis()
ax.spines[['top','right']].set_visible(False)
fig.savefig(RESULT_ROOT/'outputs/08_spec_accuracy.png',dpi=150)
plt.close(fig)

states = [('정답','#cde9d9'),('오답','#f5d5cf'),('정답 미확정','#f5e5aa'),('미평가','#e1e8f0')]
def heatmap_state(row):
    """완전 일치 및 평가 제외 사유를 색상에 매핑함."""
    if row.evaluation_status=='ground_truth_uncertain':
        return 2
    if row.evaluation_status=='not_evaluated_no_response':
        return 3
    return 0 if row.full_match else 1
matrix = np.array([heatmap_state(row) for row in detail.itertuples()]).reshape(len(targets),len(SPEC_FIELDS))
fig, ax = plt.subplots(figsize=(11.2,4.1),layout='constrained')
ax.imshow(matrix,cmap=plt.matplotlib.colors.ListedColormap([c for _,c in states]),vmin=0,vmax=3,aspect='auto')
ax.set(xticks=range(len(labels)),xticklabels=labels,yticks=range(len(targets)),
       yticklabels=[f'{maker}\n{pn}' for maker,(_,pn) in zip(makers,targets)])
ax.set_title('개발용 대표 P/N 규격별 완전 일치 판정',pad=14)
for i,j in np.ndindex(matrix.shape):
    ax.text(j,i,states[matrix[i,j]][0],va='center',ha='center',color='#243446')
ax.tick_params(length=0,pad=7)
ax.legend(handles=[Patch(color=c,label=f'{label} {int((matrix==i).sum())}개')
                   for i,(label,c) in enumerate(states)],
          ncols=4,loc='upper center',bbox_to_anchor=(.5,-.13),frameon=False)
fig.savefig(RESULT_ROOT/'outputs/08_evaluation_heatmap.png',dpi=150)
plt.close(fig)

labels_issue = {
    'scope_mismatch':'P/N 적용 범위 불일치',
    'source_type_mismatch':'근거 유형 불일치',
    'temperature_upper_wrong':'동작 온도 상한 오류',
    'pn_link_unconfirmed':'P/N 근거 연결 미확정',
    'evidence_missing':'페이지·인용 누락',
    'measurement_condition_missing':'ESR 측정 조건 누락',
    'temperature_lower_missing':'동작 온도 하한 누락',
    'range_evidence_incomplete':'전체 온도 범위 근거 누락',
}
order=list(dict.fromkeys(issues.error_type))
fig,(ax,api_ax)=plt.subplots(1,2,figsize=(11.3,5.3),layout='constrained',gridspec_kw={'width_ratios':[1.7,1]})
offset=np.zeros(len(order),dtype=int)
for maker,color in zip(makers[:2],('#385f8d','#805227')):
    counts=issues.loc[issues.manufacturer.eq(maker),'error_type'].value_counts()
    values=[int(counts.get(issue,0)) for issue in order]
    bars=ax.barh([labels_issue.get(issue,issue) for issue in order],values,left=offset,label=maker,color=color)
    ax.bar_label(bars,labels=[str(n) if n else '' for n in values],label_type='center',color='white')
    offset+=values
ax.set(title='9단계 정답 비교에서 확인된 사유',xlabel='사유 수 · 한 규격에 복수 사유 가능',
       xticks=range(max(offset)+1),xlim=(0,max(offset)+.3))
ax.legend(loc='upper center',bbox_to_anchor=(.5,-.12),frameon=False,ncols=2)
api_counts=[int(api.manufacturer.eq(maker).sum()) for maker in makers]
bars=api_ax.barh(makers,api_counts,color='#a75145')
api_ax.bar_label(bars,padding=4)
api_ax.set(title='별도 집계 · 이전 API HTTP 503',xlabel='요청 수',xticks=range(max(api_counts)+1),
           xlim=(0,max(api_counts)+.45))
for axis in (ax,api_ax):
    axis.invert_yaxis()
    axis.spines[['top','right']].set_visible(False)
fig.savefig(RESULT_ROOT/'outputs/08_error_distribution.png',dpi=150)
plt.close(fig)

## 평가 결과 저장 및 재읽기

In [7]:
tables = {
    'results/08_evaluation_detail.csv':detail,
    'results/08_evaluation_summary.csv':summary,
    'results/08_error_analysis.csv':errors,
}
for relative,table in tables.items():
    path=RESULT_ROOT/relative
    table.to_csv(path,index=False,encoding='utf-8-sig')
    reloaded=read_csv(path)
    pd.testing.assert_frame_equal(reloaded,table.fillna('').astype(str) if relative!='results/08_evaluation_summary.csv'
        else table.astype(object).where(table.notna(),'').astype(str))
for filename in ('08_spec_accuracy.png','08_evaluation_heatmap.png','08_error_distribution.png'):
    with Image.open(RESULT_ROOT/'outputs'/filename) as png:
        assert png.format=='PNG'
        png.verify()
assert hashlib.sha256(truth_path.read_bytes()).hexdigest()==truth_sha256
assert all(path.suffix.lower()=='.png' for path in (ROOT/'outputs').rglob('*') if path.is_file())
print('정답 CSV 보존, 평가 CSV 3개 재읽기, PNG 3개 형식 확인함.')

정답 CSV 보존, 평가 CSV 3개 재읽기, PNG 3개 형식 확인함.


## 평가 한계

- 대표 개발용 소표본 기준임
- Vishay 서지전압은 단위 불명확으로 정답 추가 검토 필요함
- Panasonic은 실제 응답 확보 전까지 정확도 분모에서 제외함
- 전체 20개 P/N과 평가용 데이터의 성능은 산출하지 않음